# Philodoptera Dataset

Stridulation in crickets (e.g. [Pholidoptera littoralis](https://www.orthoptera.ch/wiki/arten/ensifera/tettigoniinae/item/pholidoptera-littoralis)) is a great animal model to study motor control/ sound production. Below is a tutorial on how we can create a `trials.nc`. Note that we have a video, audio and DeepLabCut pose file, meaning we can look at **movement** and **sound** features in the GUI!

<img src="assets/cricket1.png" width="900">
<img src="assets/cricket2.png" width="900">

Made in Slovenia, 2025.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import xarray as xr
from pathlib import Path
from audioio import load_audio

from movement.io import load_poses
from movement.kinematics import compute_velocity, compute_speed, compute_acceleration

import ethograph as eto
from ethograph import get_project_root
from ethograph.utils.download import download_example_dataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Download example data


In [2]:
data_folder = get_project_root() / "data" / "philodoptera"
download_example_dataset("philodoptera", data_folder)

print(f"\ndata_folder:  {data_folder}")

  philodoptera.csv (4/4)

data_folder:  C:\Users\aksel\Documents\Code\EthoGraph\data\philodoptera


### Create dataset


In [3]:
project_root = Path.cwd().parent
os.chdir(project_root)

data_folder = eto.get_project_root() / "data" / "philodoptera"
poses_csv_path = data_folder / "philodoptera.csv"
audio_path =  data_folder/ "philodoptera.wav"
video_path = data_folder / "philodoptera.mp4"

In [4]:
fps = 240

# The "philodoptera.csv" contains position (xy) data of the left/right wing tip
ds = load_poses.from_dlc_file(poses_csv_path, fps=fps)

# Derive some kinematic features for wing position
ds["velocity"] = compute_velocity(ds.position)
ds["speed"] = compute_speed(ds.position)
ds["acceleration"] = compute_acceleration(ds.position)


In [5]:
ds['individuals'] = ["Pholidoptera_littoralis_1"]
ds.attrs["Recording info"] = "Akseli Ilmanen: This cricket species Pholidoptera littoralis was recorded on a field trip in Slovenia in 2025. The video was recorded with an iPhone and the audio with an Zoom recorder. Synchronization may not be perfect."


In [12]:
dt = eto.dataset_to_basic_trialtree(ds)

dt.set_media("video", [video_path.name], device_labels=["cam-1"])
dt.set_media("audio", [audio_path.name], device_labels=["mic-1"])
dt.set_media("pose", [poses_csv_path.name], device_labels=["cam-1"])


output_path = data_folder / "philodoptera.nc"
dt.to_netcdf(output_path)